# Cassini CDA Dust Analyzer: 3-P Focused Ablation Study (Mixed k)

This notebook conducts a specialized ablation study focusing on the **3-P** dust particle class. 
To evaluate performance under the optimal configuration, we use a mixed $k$-shot few-shot learning configuration:
- **k = 16** reference examples for all standard classes (Noise, 1, 2, 3, 4, 5, 5-Na) to cover their rich structural variations.
- **k = 4** reference examples for the rare class **3-P** (as there are only 9 total samples in the dataset, leaving 5 samples for test evaluation).

We employ a hybrid prompt structure where each reference image is accompanied by both a static **General Profile** of the class (to ground the VLM in global rules) and a dynamic **Specific Sample Features** explanation (to highlight the local characteristics of the reference spectrum).

## Step 1: Imports and Configurations

In [1]:
import os
import io
import json
import base64
import asyncio
import time
import subprocess
import pandas as pd
import numpy as np
from datetime import datetime
from google import genai
from google.genai import types
from google.cloud import storage
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.signal import savgol_filter
import huggingface_hub
import tqdm
warnings.filterwarnings("ignore")
import logging
logging.getLogger("google_genai").setLevel(logging.WARNING)
from dotenv import load_dotenv
try:
    dotenv_path = os.path.join(os.path.dirname(__file__), '../.env')
    project_root = os.path.join(os.path.dirname(__file__), '..')
except NameError:
    dotenv_path = "../.env"
    project_root = ".."
load_dotenv(dotenv_path=dotenv_path)
import sys
sys.path.append(os.path.abspath(project_root))
from cda_dust_agent.config import Config
from cda_dust_agent.tools.utils import generate_spectrum_image_bytes, parse_response
from scipy.signal import savgol_filter
import random
configs = Config()
model_id = configs.agent_settings.model


## Step 2: Custom Spectral Guidelines and Hybrid Prompts for 3-P
We define the critical differentiation guidelines for Class 3 vs Class 3-P and outline the static General Profiles used for the hybrid prompt structure.

In [2]:
SYSTEM_INSTRUCTION_TEXT = """You are an expert Cosmic Dust Spectroscopist analyzing Cassini Cosmic Dust Analyzer (CDA) time-of-flight mass spectra.
You will be provided with images of 1D spectra plotted on a logarithmic y-axis (signal amplitude) and a linear x-axis (time-of-flight index).

---

### CRITICAL SPECTRAL BEHAVIOR & SHIFTING:
- These spectra can be shifted in time by up to 50 index points due to hardware trigger recording differences.
- Because of the non-linear mapping between time-of-flight and mass, this shift causes the peaks to visually stretch.
- Do NOT rely on absolute x-axis index positions. Instead, focus on relative shapes, relative peak sequences, and overall topography.

---

### CRITICAL DIFFERENTIATION GUIDE FOR CLASS 3 VS CLASS 3-P:
When a spectrum exhibits broad organic-like envelopes or carbon cluster humps, you must distinguish between Class 3 (General Organics) and Class 3-P (Burst-and-Trail Agglomerate):
1. **Class 3 (General Organics):** Features multiple broad, asymmetric "shark-fin" peak clusters with expanding periodicity (representing carbon series). Crucially, on the scaled [0, 1] y-axis, the valleys between clusters drop significantly lower (down to y < 0.15), and it lacks a singular early maximum (index 200) that dominates the entire spectrum by a factor of 10.
2. **Class 3-P (Burst-and-Trail Agglomerate):** Dominated by an explosive primary complex (index 190-220) which is the absolute global maximum (y = 1.0), and a distinct secondary peak at index 330-345. Crucially, past index 400 it rests on a continuous, flat "chemical noise" plateau (baseline cushion) that remains highly elevated (y ≈ 0.3 to 0.5) and never drops back to y ≈ 0, before dropping off sharply into the noise floor around index 650-750.

---

### CLASS SPECIFIC PROFILES:

#### Class Noise
- Devoid of Gaussian peaks, elemental clusters, or chemical spacing.
- Characterized by a narrow, high-intensity initial trigger spike (often index 10-20), followed by a broad envelope of high-frequency digitizer noise ("grass").
- Terminates abruptly at an electronic cutoff (index 640-850), returning to a flat baseline with rare, single-point dark counts.

#### Class 1
- Pure water ice spectrum consisting of a regular sequence of hydronium cluster peaks ($H_3O^+(H_2O)_n$) at mass 19, 37, 55, 73, 91... (index locations ~85, ~120, ~146, ~169, ~189).
- Early global maximum (index 80-100) followed by a rapid, step-like decay of peak heights towards the right.
- Valleys between peaks return completely to the flat baseline (no intermediate peak structures or elevated organic noise).
- Terminates in a sharp cutoff near index 650.

#### Class 2
- Organic-bearing or dirty water ice.
- Features the same hydronium cluster peak locations as Class 1, but has significant organic/saline contamination.
- Valleys between the major peaks are filled (elevated signal baseline) with unresolved organic background, secondary peaks, or high-frequency fluctuations.
- Global maximum is often in the index 180-300 range, showing massive, broad, unresolved molecular cluster sequences.
- Terminates in a sheer drop-off cliff around index 640-700.

#### Class 3
- Characterized by a continuous, highly elevated "mesa" plateau or unresolved complex organic mixture.
- Does not return to baseline between peaks throughout the spectrum (sustained signal above the noise floor).
- Typically features 3 to 4 broad, asymmetric "shark-fin" peak clusters with expanding periodicity (representing carbon clusters $C_n$ or heavy homologous organic series). Water peaks are absent or negligible.

#### Class 4
- Mineral/silicate-rich spectrum.
- Characterized by isolated, needle-sharp atomic spikes in the low-mass region (like $Mg^+$ at 24 Da, $Si^+$ at 28 Da, $Fe^+$ at 56 Da) with a very quiet baseline in between.
- Crucially, lacks the repeating, comb-like water cluster sequence ($H_3O^+(H_2O)_n$) of Class 1 and 2.
- Transitioning to broad mid-mass envelopes with a global maximum at index 280-350 and a distinct late cluster around 420-480, terminating in a hard cutoff near index 650.

#### Class 5
- Characterized by a bimodal extreme: an overwhelmingly intense primary peak in the early region (index 60-90, peaking near 70-75) with a long, trailing decay edge.
- The rest of the spectrum is a dense, uniform, low-amplitude "barcode" or "grass" band that crashes into a hard cliff at index 650.

#### Class 5-Na
- Bipartite structure dominated by Sodium chemistry.
- Erupts with a singular, overwhelmingly intense, sharp spike (the Sodium payload) at index 135-160.
- Followed by a delayed, prolonged, highly noisy, and completely unresolved plateau from index 200 to 650 (representing detector saturation, plasma shielding, or complex sodium-water clusters) that terminates in a hard cliff.

#### Class 3-P
- Characterized by a "burst-and-trail" signature consisting of broad, unresolved mass envelopes rather than sharp, isolated single-element lines.
- Features a massive primary complex (index ~200-220) which is the absolute maximum, followed by a distinct secondary peak around index ~330-345.
- Displays a unique elevated baseline/plateau past index 400 that never returns to zero (chemical noise plateau), with superimposed broad rhythmic hummocks, terminating in a rapid collapse/cutoff between index 650 and 750.

---

### CLASSIFICATION TASK:
Analyze the provided target spectrum image. Compare it to the reference examples and class profiles. Output your prediction using one of the following exact labels:
- "1"
- "2"
- "3"
- "4"
- "5"
- "5-Na"
- "3-P"
- "Noise"

Return the prediction strictly in the requested JSON format."""

CLASSIFICATION_USER_PROMPT = """Based on the provided examples, classify this new spectrum.
Carefully compare its visual features (peaks, baseline, noise levels, and overall structure) to the examples, keeping in mind that peak shifts and amplitude variations can occur within the same class.

Return your analysis strictly in the following JSON format:
{
    "id": "<the provided sclk id>",
    "class": "<the predicted class label>",
    "explanation": "<a brief 1-2 sentence explanation of why it belongs to this class based on visual features>"
}
"""

# Prompts for Stage 2 (VLM Self-Correction Loop for Class 3 vs Class 3-P)
SYSTEM_INSTRUCTION_TEXT_SELF_CORRECT = """You are an expert Cosmic Dust Spectroscopist analyzing Cassini Cosmic Dust Analyzer (CDA) time-of-flight mass spectra.
Your task is to review and verify a first-pass prediction for a spectrum that was classified as either Class 3 (General Organics) or Class 3-P (Burst-and-Trail Agglomerate).

There is a common misclassification trap where VLMs misidentify a noisy or cluster-rich Class 3 baseline as a Class 3-P plateau. You must rigorously verify the prediction by evaluating these three criteria:

1. **Primary Peak Dominance:**
   - Class 3-P: Must have a single, explosive primary peak complex (index ~200-220) that is the absolute global maximum (y = 1.0) and dominates the rest of the spectrum by a factor of 5 to 10.
   - Class 3: Has multiple broad peak clusters (carbon chain clusters) of relatively comparable heights across the index range, without a single early maximum dominating the entire spectrum.

2. **Valley Depth between Clusters:**
   - Class 3: The valleys between the broad peak clusters drop significantly lower, returning close to the noise floor (down to y < 0.15).
   - Class 3-P: The signal remains highly elevated throughout.

3. **Baseline Plateau Cushion (past index 400):**
   - Class 3-P: Crucially, past index 400, the signal rests on a continuous, flat "chemical noise" plateau (baseline cushion) that remains highly elevated (y ≈ 0.3 to 0.5) and never returns to zero before dropping off sharply near index 650-750.
   - Class 3: The baseline past index 400 decays back close to zero.

You will be provided with:
- The target spectrum image.
- Reference examples of true Class 3 and Class 3-P spectra.

Analyze the image, critique the first-pass reasoning, and output your final prediction (either "3" or "3-P")."""

CLASSIFICATION_USER_PROMPT_SELF_CORRECT = """Here is the first-pass classification for the target spectrum:
- First-pass Predicted Class: {first_pass_class}
- First-pass Explanation: {first_pass_explanation}

Review the target spectrum image carefully against the reference examples of Class 3 and Class 3-P.
Perform a self-correction check. Specifically, verify if the first-pass prediction fell into the trap of misidentifying a Class 3 spectrum with multiple broad peaks or baseline noise as Class 3-P.

Return your analysis strictly in this JSON format:
{{
    "id": "{sclk}",
    "first_pass_analysis_critique": "<your critique of the first-pass explanation, explaining if it fell into the Class 3 baseline noise trap>",
    "class": "<your final prediction: '3' or '3-P'>",
    "explanation": "<your final 1-2 sentence explanation supporting the final prediction>"
}}
"""

ANNOTATION_USER_PROMPT = """This is a time-of-flight mass spectrum for a particle belonging to the known class '{label}'.
Please provide a brief, 1-2 sentence description of the key visual features that characterize this spectrum as '{label}'. Do not output JSON, just the text description."""

CONTRASTIVE_ANNOTATION_USER_PROMPT = """You are an expert Cosmic Dust Spectroscopist.
We want you to write a brief, 1-2 sentence description of the key visual features of the Target Spectrum (labeled as Class '{label}') to be used as a reference example for class '{label}'.

To help you write a contrastive explanation that clearly distinguishes '{label}' from all other classes, we have provided the Target Spectrum (Image A) alongside reference spectra from the other classes.

Please review all the provided images:
- Image A (Target): This is the spectrum of '{label}' you must describe.
{reference_descriptions}

Key Spectral Guidelines to remember:
- Class 1: Pure water ice. Clear, sharp sequence of hydronium cluster peaks ($H_3O^+(H_2O)_n$) at mass 19, 37, 55, 73... (index locations ~85, ~120, ~146, ~169, ~189). Global maximum is early (~80-100). Valleys between peaks return fully to baseline.
- Class 2: Organic-rich water ice. Same hydronium peaks as Class 1, but with significant valley-filling, organic background, or intermediate peaks between them. Global maximum is often in the 180-300 range.
- Class 4: Mineral spectrum. Needle-sharp atomic spikes (Mg+, Si+, Fe+) with a quiet baseline, and completely lacks the repeating water-ice cluster sequence. Mid-mass envelopes with global maximum at index 280-350 and a distinct late cluster around index 420-480.
- Class 3: Organics. Multiple broad asymmetric shark-fin peak clusters representing carbon clusters. Crucially, on the scaled [0, 1] y-axis, the valleys between clusters drop significantly lower (down to y < 0.15), and it lacks the massive early maximum (index 200) and the flat, continuous chemical noise plateau past index 400 seen in Class 3-P.
- Class 5: Bimodal extreme (overwhelmingly intense early peak with long trailing decay, rest is dense low-amplitude barcode noise).
- Class 5-Na: Singular sharp Sodium payload peak at index 135-160, followed by a delayed, noisy detector-saturation plateau.
- Class 3-P: Burst-and-trail signature. Massive primary peak complex (index ~200-220) as the global maximum (y = 1.0), a distinct secondary peak at index ~330-345, and an elevated baseline plateau past index 400 that remains highly elevated (y ≈ 0.3 to 0.5) and never returns to zero.
- Noise: Narrow initial trigger spike followed by broad envelope of digitizer noise ("grass"), devoid of chemical peaks.

Write a 1-2 sentence description of the visual features of Image A (Target) that characterize it as '{label}', highlighting specific details (such as peak spacing trend, peak shape, or noise baseline) that help distinguish it from the other classes. Do not output JSON, just return the text description."""

# General profiles for hybrid prompt logic
GENERAL_PROFILES = {
    "Noise": "Instrumental digitizer noise showing a narrow trigger spike and high-frequency 'grass', completely devoid of chemical peaks.",
    "1": "Pure water ice sequence of regularly spaced hydronium cluster peaks ($H_3O^+(H_2O)_n$) with an early global maximum and valleys returning fully to baseline.",
    "2": "Organic-bearing water ice featuring hydronium peaks similar to Class 1, but with prominent organic background noise filling the valleys between peaks.",
    "3": "General organic chains consisting of multiple broad, asymmetric 'shark-fin' peak clusters (carbon chains), lacking a singular early dominant peak.",
    "4": "Mineral/silicate spectrum showing isolated, needle-sharp atomic spikes (Mg+, Si+, Fe+) with a very quiet, flat baseline in between.",
    "5": "Bimodal extreme featuring an overwhelmingly intense primary peak in the early region (~70-75) with a long trailing decay and a uniform low-amplitude barcode band.",
    "5-Na": "Bipartite sodium chemistry dominated by a singular, intense early sodium spike (~135-160) followed by a delayed, noisy detector-saturation plateau.",
    "3-P": "Burst-and-trail signature showing an explosive early maximum (~200), a distinct secondary peak (~340), and a continuous elevated chemical noise baseline past index 400."
}

## Step 3: Fetching and Preprocessing Data

In [3]:
def download_and_preprocess_data():
    from scipy.signal import savgol_filter
    print("Downloading train dataset from Hugging Face...")
    REPO_ID = "CosmicDustGroup/cassini-cda-spectra"
    FILENAME_TRAIN = "data/lvl2/cda_qm_spectra_pre2008277_train_lvl2.parquet"
    
    file_train_path = huggingface_hub.hf_hub_download(repo_id=REPO_ID, filename=FILENAME_TRAIN, repo_type="dataset")
    print(f"Downloaded train dataset to: {file_train_path}")
    
    df = pd.read_parquet(file_train_path)
    print(f"Loaded {len(df)} raw spectra.")
    
    # Filter 1018 length spectra
    df = df[df['spectrum'].apply(len) == 1018].copy()
    print(f"Filtered to {len(df)} spectra of length 1018.")
    
    # Crop spectrum to index 10 to 640
    def crop_spectrum(spectrum):
        return spectrum[10:641]
    df['spectrum'] = df['spectrum'].apply(crop_spectrum)
    
    # Map labels:
    # Noise -> Noise
    # 1 -> 1
    # 2 -> 2
    # 3-P -> 3-P
    # 5-Na -> 5-Na
    # 5 -> 5
    # 4 -> 4
    # sub-classes of 3 other than 3-P -> 3
    # ?, X, 2-X -> ?
    def map_label(x):
        if not isinstance(x, str):
            return "?"
        if x in ["?", "X", "2-X"]:
            return "?"
        if x == "3-P":
            return "3-P"
        if x.startswith("3-") or x == "3":
            return "3"
        return x
        
    df['class'] = df['class'].apply(map_label)
    
    # Exclude "?" (inference class)
    df = df[df['class'] != '?'].copy()
    print(f"Remaining active spectra after mapping and filtering: {len(df)}")
    
    # Scaling and smoothing
    def qm_scaling_savgol(spectrum):
        spectrum = np.array(spectrum, dtype=float)
        # Apply log10
        log_spec = np.log10(spectrum + np.abs(np.min(spectrum)))
        
        # Replace negative infinity/NaN with minimum finite value
        finite_mask = np.isfinite(log_spec)
        if np.any(finite_mask):
            min_finite_val = np.min(log_spec[finite_mask])
        else:
            min_finite_val = 0
        log_spec = np.nan_to_num(log_spec, neginf=min_finite_val, nan=min_finite_val)
        
        # Min-max scaling
        spec_min = np.min(log_spec)
        spec_max = np.max(log_spec)
        range_val = spec_max - spec_min
        if range_val > 0:
            scaled_spectrum = (log_spec - spec_min) / range_val
        else:
            scaled_spectrum = np.zeros_like(log_spec)
            
        return savgol_filter(scaled_spectrum, 5, 3)
        
    df['spectrum'] = df['spectrum'].apply(qm_scaling_savgol)
    print("Scaling and smoothing complete.")
    
    return df

# 2. Split train/test

df = download_and_preprocess_data()


2026-06-28 22:24:54,976 - [httpx] - INFO - HTTP Request: HEAD https://huggingface.co/datasets/CosmicDustGroup/cassini-cda-spectra/resolve/main/data/lvl2/cda_qm_spectra_pre2008277_train_lvl2.parquet "HTTP/1.1 302 Found"
2026-06-28 22:24:54,977 - [huggingface_hub.utils._http] - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Downloaded train dataset to: /root/.cache/huggingface/hub/datasets--CosmicDustGroup--cassini-cda-spectra/snapshots/e45dfccb305fffab4b468a7d5a06b38dda92cba4/data/lvl2/cda_qm_spectra_pre2008277_train_lvl2.parquet
Loaded 18883 raw spectra.
Filtered to 18883 spectra of length 1018.
Remaining active spectra after mapping and filtering: 18325
Scaling and smoothing complete.


## Step 4: Partitioning Data into Train and Test Pools
We sample exactly 16 training samples for standard classes and 4 training samples for Class 3-P, with up to 100 test samples per class from the remaining pool.

In [4]:
def split_train_test(df):
    train_sclks = {}
    sampled_rows = []
    
    classes = ['Noise', '1', '2', '3', '4', '5', '5-Na', '3-P']
    
    for cls in classes:
        cls_df = df[df['class'] == cls]
        # We need exactly 16 training samples for standard classes, 4 for 3-P
        n_train = 4 if cls == '3-P' else 16
        cls_train = cls_df.sample(n=n_train, random_state=123)
        train_sclks[cls] = cls_train['sclk'].tolist()
        sampled_rows.append(cls_train)
        
    train_df = pd.concat(sampled_rows, ignore_index=True)
    all_train_sclks = [sclk for sclks in train_sclks.values() for sclk in sclks]
    
    test_pool = df[~df['sclk'].isin(all_train_sclks)].copy()
    
    TEST_SIZE_PER_CLASS = 100
    test_dfs = []
    for cls in classes:
        cls_test_pool = test_pool[test_pool['class'] == cls]
        n_samples = min(TEST_SIZE_PER_CLASS, len(cls_test_pool))
        cls_test = cls_test_pool.sample(n=n_samples, random_state=42)
        test_dfs.append(cls_test)
        print(f"Included class '{cls}' in test set with {len(cls_test)} samples.")
        
    test_df = pd.concat(test_dfs, ignore_index=True)
    print(f"Train samples: {len(train_df)} (7 classes x 16 + 1 class x 4)")
    print(f"Test samples: {len(test_df)}")
    
    return train_df, test_df, train_sclks

# 3. Generate VLM explanations concurrently

train_df, test_df, train_sclks = split_train_test(df)


Included class 'Noise' in test set with 100 samples.
Included class '1' in test set with 100 samples.
Included class '2' in test set with 100 samples.
Included class '3' in test set with 100 samples.
Included class '4' in test set with 100 samples.
Included class '5' in test set with 100 samples.
Included class '5-Na' in test set with 22 samples.
Included class '3-P' in test set with 5 samples.
Train samples: 116 (7 classes x 16 + 1 class x 4)
Test samples: 627


## Step 5: Generating VLM Explanations concurrently
We generate online annotations using Gemini.

In [5]:
async def generate_explanation(client, model_id, spectrum_array, label, sclk, semaphore, references=None):
    async with semaphore:
        def format_class_label(cls):
            cls_str = str(cls)
            if cls_str.lower().startswith("class"):
                return cls_str
            return f"Class {cls_str}" if cls_str.lower() != 'noise' else "Noise"

        target_label = format_class_label(label)
        img_bytes = generate_spectrum_image_bytes(np.array(spectrum_array), title=f"{target_label} Sample {sclk}")
        
        if references:
            ref_descriptions = ""
            for idx, (other_cls, _) in enumerate(references):
                char_code = chr(66 + idx) # B, C, D, E...
                ref_descriptions += f"- Image {char_code}: Reference Spectrum of '{format_class_label(other_cls)}'\\n"
                
            prompt = CONTRASTIVE_ANNOTATION_USER_PROMPT.format(
                label=target_label,
                reference_descriptions=ref_descriptions.strip()
            )
            
            parts = [
                types.Part.from_text(text=prompt),
                types.Part.from_bytes(data=img_bytes, mime_type="image/png") # Image A
            ]
            for _, other_img_bytes in references:
                parts.append(types.Part.from_bytes(data=other_img_bytes, mime_type="image/png"))
                
            contents = [types.Content(role="user", parts=parts)]
        else:
            prompt = ANNOTATION_USER_PROMPT.format(label=label)
            contents = [
                types.Content(role="user", parts=[
                    types.Part.from_text(text=prompt),
                    types.Part.from_bytes(data=img_bytes, mime_type="image/png")
                ])
            ]
        
        retries = 3
        for attempt in range(retries):
            try:
                response = await client.aio.models.generate_content(
                    model=model_id,
                    contents=contents,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_INSTRUCTION_TEXT
                    )
                )
                return {
                    "label": label,
                    "sclk": sclk,
                    "image": img_bytes,
                    "explanation": response.text.strip(),
                    "image_base64": base64.b64encode(img_bytes).decode('utf-8')
                }
            except Exception as e:
                print(f"Failed to generate explanation for {label} (sclk: {sclk}): {e}. Retrying {attempt+1}/{retries}...")
                await asyncio.sleep(2 ** attempt)
        return None

async def generate_explanations_pool(client, model_id, train_df, train_sclks):
    print("Pre-rendering and caching training spectrum images...")
    image_cache = {}
    for _, row in train_df.iterrows():
        sclk = row['sclk']
        cls = row['class']
        label = f"Class {cls}" if str(cls).lower() != 'noise' else "Noise"
        img_bytes = generate_spectrum_image_bytes(np.array(row['spectrum']), title=f"{label} Reference Sample {sclk}")
        image_cache[sclk] = img_bytes
        
    api_semaphore = asyncio.Semaphore(10)
    explanation_tasks = []
    
    import random
    random.seed(123)
    
    print(f"Generating explanations for {len(train_df)} training samples concurrently with dynamic contrastive references...")
    for _, row in train_df.iterrows():
        target_sclk = row['sclk']
        target_cls = row['class']
        
        references = []
        for other_cls, other_sclks in train_sclks.items():
            if other_cls != target_cls:
                other_sclk = random.choice(other_sclks)
                references.append((other_cls, image_cache[other_sclk]))
                
        task = generate_explanation(client, model_id, row['spectrum'], row['class'], row['sclk'], api_semaphore, references=references)
        explanation_tasks.append(task)
            
    results = await asyncio.gather(*explanation_tasks)
    explanations_pool = [r for r in results if r is not None]
    print(f"Successfully generated {len(explanations_pool)} explanations.")
    return explanations_pool

# 4. Create consolidated batch input file

# Configure the client wrapper
configs = Config()
client = genai.Client()
model_id = configs.agent_settings.model
explanations_pool = await generate_explanations_pool(client, model_id, train_df, train_sclks)


Pre-rendering and caching training spectrum images...
Generating explanations for 116 training samples concurrently with dynamic contrastive references...
Successfully generated 116 explanations.


## Step 6: Create Consolidated Batch Input File (Mixed k)
We compile requests into a single JSONL file using the hybrid prompt logic (General Profile + Specific Sample Features), mapping $k=4$ for `3-P` and $k=16$ for other classes.

In [6]:
def create_ablation_batch_input_file(test_df, explanations_pool, k_values, output_file):
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    print(f"Generating consolidated batch request file for k={k_values}...")
    
    print("Pre-rendering and caching target spectrum images...")
    test_image_cache = {}
    fig, ax = plt.subplots(figsize=(12, 6), dpi=60)
    
    for idx, row in test_df.reset_index(drop=True).iterrows():
        sclk = row['sclk']
        if sclk not in test_image_cache:
            ax.clear()
            ax.plot(row['spectrum'], color='black', linewidth=1.5)
            ax.set_title(f"Sample {sclk}")
            ax.set_ylim(-0.02, 1.02)
            ax.set_xlim(0, 630)
            ax.grid(True)
            
            buf = io.BytesIO()
            fig.savefig(buf, format='png')
            buf.seek(0)
            img_bytes = buf.getvalue()
            img_b64 = base64.b64encode(img_bytes).decode('utf-8')
            test_image_cache[sclk] = img_b64
            
        if (idx + 1) % 100 == 0 or (idx + 1) == len(test_df):
            print(f"  -> Pre-rendered {idx + 1}/{len(test_df)} images...")
            
    plt.close(fig)
    
    requests = []
    class_explanations = {}
    for ex in explanations_pool:
        cls = ex['label']
        if cls not in class_explanations:
            class_explanations[cls] = []
        class_explanations[cls].append(ex)
        
    for k in k_values:
        print(f"  -> Building request payloads for k={k} shots (with 4 for 3-P and 16 for other classes)...")
        k_shot_examples = []
        for cls, exs in class_explanations.items():
            limit = 4 if cls == "3-P" else 16
            k_shot_examples.extend(exs[:limit])
            
        used_sclk_ids = [ex['sclk'] for ex in k_shot_examples]
        
        few_shot_parts = [{"text": CLASSIFICATION_USER_PROMPT}]
        if k_shot_examples:
            few_shot_parts.append({"text": "Here are reference examples:"})
            for ex in k_shot_examples:
                label_key = ex['label']
                gen_profile = GENERAL_PROFILES.get(label_key, "")
                class_label = f"Class {label_key}" if str(label_key).lower() != 'noise' else "Noise"
                few_shot_parts.append({"text": f"Example: {class_label}\\n- General Profile: {gen_profile}\\n- Specific Sample Features: {ex['explanation']}"})
                few_shot_parts.append({"inline_data": {"mime_type": "image/png", "data": ex['image_base64']}})
            few_shot_parts.append({"text": "Now, analyze the following spectrum:"})
            
        for _, row in test_df.iterrows():
            if row['sclk'] in used_sclk_ids:
                continue
                
            img_b64 = test_image_cache[row['sclk']]
            
            current_parts = few_shot_parts.copy()
            current_parts.append({"inline_data": {"mime_type": "image/png", "data": img_b64}})
            current_parts.append({"text": f"Sample ID (sclk): {row['sclk']} | k: {k}"})
            
            request = {
                "request": {
                    "contents": [
                        {
                            "role": "user",
                            "parts": current_parts
                        }
                    ],
                    "systemInstruction": {
                        "parts": [{"text": SYSTEM_INSTRUCTION_TEXT}]
                    },
                    "generationConfig": {
                        "responseMimeType": "application/json",
                        "responseSchema": {
                            "type": "OBJECT",
                            "properties": {
                                "id": {"type": "STRING", "description": "The ID of the run (sclk)"},
                                "class": {"type": "STRING", "description": "The predicted class label"},
                                "explanation": {"type": "STRING", "description": "Explanation for the prediction"},
                                "k": {"type": "INTEGER", "description": "The value of k shots used"}
                            },
                            "required": ["id", "class", "explanation", "k"]
                        }
                    }
                }
            }
            requests.append(json.dumps(request))
            
    with open(output_file, 'w') as f:
        for req in requests:
            f.write(req + '\n')
            
    print(f"Generated JSONL file with {len(requests)} requests at {output_file}")
    return output_file

def create_self_correction_batch_input_file(reclass_df, explanations_pool, output_file):
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    print(f"Generating Stage 2 self-correction request file for {len(reclass_df)} samples...")
    
    print("Pre-rendering and caching target spectrum images...")
    test_image_cache = {}
    fig, ax = plt.subplots(figsize=(12, 6), dpi=60)
    
    for idx, row in reclass_df.reset_index(drop=True).iterrows():
        sclk = row['sclk']
        if sclk not in test_image_cache:
            ax.clear()
            ax.plot(row['spectrum'], color='black', linewidth=1.5)
            ax.set_title(f"Sample {sclk}")
            ax.set_ylim(-0.02, 1.02)
            ax.set_xlim(0, 630)
            ax.grid(True)
            
            buf = io.BytesIO()
            fig.savefig(buf, format='png')
            buf.seek(0)
            img_bytes = buf.getvalue()
            img_b64 = base64.b64encode(img_bytes).decode('utf-8')
            test_image_cache[sclk] = img_b64
            
        if (idx + 1) % 100 == 0 or (idx + 1) == len(reclass_df):
            print(f"  -> Pre-rendered {idx + 1}/{len(reclass_df)} images...")
            
    plt.close(fig)
    
    # Extract training examples only for Class 3 and 3-P
    class_explanations = {}
    for ex in explanations_pool:
        cls = ex['label']
        if cls not in class_explanations:
            class_explanations[cls] = []
        class_explanations[cls].append(ex)
        
    k_shot_examples = []
    # 4 shots for Class 3 and 4 shots for Class 3-P
    for cls in ["3", "3-P"]:
        exs = class_explanations.get(cls, [])
        k_shot_examples.extend(exs[:4])
        
    used_sclk_ids = [ex['sclk'] for ex in k_shot_examples]
    
    few_shot_parts = []
    if k_shot_examples:
        few_shot_parts.append({"text": "Here are reference examples of Class 3 and Class 3-P spectra:"})
        for ex in k_shot_examples:
            label_key = ex['label']
            class_label = f"Class {label_key}"
            few_shot_parts.append({"text": f"Example: {class_label}\\n- Specific Sample Features: {ex['explanation']}"})
            few_shot_parts.append({"inline_data": {"mime_type": "image/png", "data": ex['image_base64']}})
            
    requests = []
    for _, row in reclass_df.iterrows():
        if row['sclk'] in used_sclk_ids:
            continue
            
        img_b64 = test_image_cache[row['sclk']]
        
        current_parts = [
            {"inline_data": {"mime_type": "image/png", "data": img_b64}}
        ]
        if few_shot_parts:
            current_parts.extend(few_shot_parts)
            
        # Format the critique user prompt, cleaning any potential curly braces in first-pass explanations
        exp_clean = row['explanation'].replace("{", "(").replace("}", ")")
        critique_prompt = CLASSIFICATION_USER_PROMPT_SELF_CORRECT.format(
            sclk=row['sclk'],
            first_pass_class=row['predicted_label'],
            first_pass_explanation=exp_clean
        )
        current_parts.append({"text": critique_prompt})
        
        request = {
            "request": {
                "contents": [
                    {
                        "role": "user",
                        "parts": current_parts
                    }
                ],
                "systemInstruction": {
                    "parts": [{"text": SYSTEM_INSTRUCTION_TEXT_SELF_CORRECT}]
                },
                "generationConfig": {
                    "responseMimeType": "application/json",
                    "responseSchema": {
                        "type": "OBJECT",
                        "properties": {
                            "id": {"type": "STRING", "description": "The ID of the run (sclk)"},
                            "first_pass_analysis_critique": {"type": "STRING", "description": "Critique of the first-pass prediction"},
                            "class": {"type": "STRING", "description": "The final predicted class label, either '3' or '3-P'"},
                            "explanation": {"type": "STRING", "description": "Final explanation for the prediction"}
                        },
                        "required": ["id", "first_pass_analysis_critique", "class", "explanation"]
                    }
                }
            }
        }
        requests.append(json.dumps(request))
        
    with open(output_file, 'w') as f:
        for req in requests:
            f.write(req + '\n')
            
    print(f"Generated JSONL file with {len(requests)} requests at {output_file}")
    return output_file

k_values = [4]
output_dir = "study_results_3p_self_correct"
local_jsonl = os.path.join(output_dir, "ablation_requests_3p.jsonl")
create_ablation_batch_input_file(test_df, explanations_pool, k_values, local_jsonl)

Generating consolidated batch request file for k=[4]...
Pre-rendering and caching target spectrum images...
  -> Pre-rendered 100/627 images...
  -> Pre-rendered 200/627 images...
  -> Pre-rendered 300/627 images...
  -> Pre-rendered 400/627 images...
  -> Pre-rendered 500/627 images...
  -> Pre-rendered 600/627 images...
  -> Pre-rendered 627/627 images...
  -> Building request payloads for k=4 shots (with 4 for 3-P and 16 for other classes)...
Generated JSONL file with 627 requests at study_results_3p_self_correct/ablation_requests_3p.jsonl


'study_results_3p_self_correct/ablation_requests_3p.jsonl'

## Step 7: Uploading and Submitting Vertex AI Batch Prediction Job

In [7]:
def submit_and_poll_batch_job(configs, local_jsonl_path):
    project_id = configs.agent_settings.project_id
    bucket_name = configs.agent_settings.bucket_name
    
    filename = os.path.basename(local_jsonl_path)
    print(f"Uploading {local_jsonl_path} to GCS bucket: {bucket_name}")
    storage_client = storage.Client(project=project_id)
    bucket = storage_client.bucket(bucket_name.replace("gs://", ""))
    
    gcs_input_path = f"input/{filename}"
    blob = bucket.blob(gcs_input_path)
    
    class ProgressFileWrapper(object):
        def __init__(self, fileobj, total_size):
            self.fileobj = fileobj
            self.total_size = total_size
            self.pbar = tqdm.tqdm(
                total=total_size,
                unit='B',
                unit_scale=True,
                desc="Uploading GCS",
                leave=True
            )
            self.bytes_read = 0

        def read(self, size=-1):
            chunk = self.fileobj.read(size)
            if chunk:
                self.bytes_read += len(chunk)
                self.pbar.update(len(chunk))
            return chunk

        def seek(self, offset, whence=0):
            self.fileobj.seek(offset, whence)
            current_pos = self.fileobj.tell()
            self.pbar.n = current_pos
            self.pbar.refresh()

        def tell(self):
            return self.fileobj.tell()

        def close(self):
            self.pbar.close()
            self.fileobj.close()

    total_size = os.path.getsize(local_jsonl_path)
    with open(local_jsonl_path, 'rb') as f:
        wrapped_file = ProgressFileWrapper(f, total_size)
        blob.chunk_size = 10 * 1024 * 1024
        blob.upload_from_file(wrapped_file, content_type="application/json", timeout=600)
        
    gcs_source = f"gs://{bucket.name}/{gcs_input_path}"
    print(f"Uploaded to {gcs_source}")
    
    access_token = os.popen("gcloud auth application-default print-access-token").read().strip()
    job_display_name = f"cda-3p-ablation-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    
    global_batch_req = {
        "displayName": job_display_name,
        "model": f"publishers/google/models/{configs.agent_settings.model}",
        "inputConfig": {
            "instancesFormat": "jsonl",
            "gcsSource": {"uris": [gcs_source]}
        },
        "outputConfig": {
            "predictionsFormat": "jsonl",
            "gcsDestination": {"outputUriPrefix": f"gs://{bucket.name}/output"}
        }
    }
    
    req_file = f"batch_request_3p_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(req_file, "w") as f:
        json.dump(global_batch_req, f)
        
    curl_command = [
        "curl", "-s", "-X", "POST",
        f"https://aiplatform.googleapis.com/v1/projects/{project_id}/locations/global/batchPredictionJobs",
        "-H", f"Authorization: Bearer {access_token}",
        "-H", "Content-Type: application/json; charset=utf-8",
        "-d", f"@{req_file}"
    ]
    
    print("Submitting Batch Job to Vertex AI...")
    result = subprocess.run(curl_command, capture_output=True, text=True)
    
    if os.path.exists(req_file):
        os.remove(req_file)
        
    if result.returncode == 0 and "name" in result.stdout:
        response_json = json.loads(result.stdout)
        job_name = response_json.get("name")
        print(f"Job Submitted! Name: {job_name}")
    else:
        print(f"Error submitting job: {result.stderr}\\nResponse: {result.stdout}")
        raise RuntimeError("Failed to submit Batch Prediction Job.")
        
    check_url = f"https://aiplatform.googleapis.com/v1/{job_name}"
    print("Polling job status every 30 seconds...")
    while True:
        access_token = os.popen("gcloud auth application-default print-access-token").read().strip()
        check_cmd = [
            "curl", "-s", "-X", "GET",
            check_url,
            "-H", f"Authorization: Bearer {access_token}"
        ]
        check_res = subprocess.run(check_cmd, capture_output=True, text=True)
        if check_res.returncode != 0:
            print(f"Error checking status: {check_res.stderr}")
            time.sleep(30)
            continue
            
        status_data = json.loads(check_res.stdout)
        state = status_data.get("state", "UNKNOWN")
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Job State: {state}")
        
        if state == "JOB_STATE_SUCCEEDED":
            print("Job Completed Successfully!")
            return status_data
        elif state in ["JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_PAUSED"]:
            err_msg = status_data.get("error", "No error details.")
            raise RuntimeError(f"Job ended with state {state}. Details: {err_msg}")
            
        time.sleep(30)

# --- STAGE 1: Multiclass Batch Prediction ---
print("\n--- STAGE 1: Multiclass Batch Prediction ---")
job_status = submit_and_poll_batch_job(configs, local_jsonl)


--- STAGE 1: Multiclass Batch Prediction ---
Uploading study_results_3p_self_correct/ablation_requests_3p.jsonl to GCS bucket: turansgenaibb


Uploading GCS: 100%|██████████| 3.41G/3.41G [10:21<00:00, 5.10MB/s]

Uploaded to gs://turansgenaibb/input/ablation_requests_3p.jsonl
Submitting Batch Job to Vertex AI...
Job Submitted! Name: projects/355771430623/locations/global/batchPredictionJobs/3291056280113774592
Polling job status every 30 seconds...
[22:37:35] Job State: JOB_STATE_PENDING


Uploading GCS: 100%|██████████| 3.41G/3.41G [10:40<00:00, 5.10MB/s]

[22:38:06] Job State: JOB_STATE_PENDING
[22:38:38] Job State: JOB_STATE_PENDING
[22:39:09] Job State: JOB_STATE_PENDING
[22:39:40] Job State: JOB_STATE_PENDING
[22:40:12] Job State: JOB_STATE_PENDING
[22:40:43] Job State: JOB_STATE_PENDING
[22:41:14] Job State: JOB_STATE_PENDING
[22:41:46] Job State: JOB_STATE_PENDING
[22:42:17] Job State: JOB_STATE_PENDING
[22:42:49] Job State: JOB_STATE_PENDING
[22:43:20] Job State: JOB_STATE_PENDING
[22:43:52] Job State: JOB_STATE_PENDING
[22:44:23] Job State: JOB_STATE_PENDING
[22:44:54] Job State: JOB_STATE_QUEUED
[22:45:26] Job State: JOB_STATE_RUNNING
[22:45:57] Job State: JOB_STATE_RUNNING
[22:46:29] Job State: JOB_STATE_RUNNING
[22:47:00] Job State: JOB_STATE_RUNNING
[22:47:32] Job State: JOB_STATE_RUNNING
[22:48:03] Job State: JOB_STATE_RUNNING
[22:48:35] Job State: JOB_STATE_RUNNING
[22:49:06] Job State: JOB_STATE_RUNNING
[22:49:37] Job State: JOB_STATE_RUNNING
[22:50:08] Job State: JOB_STATE_RUNNING
[22:50:40] Job State: JOB_STATE_RUNNING
[

Uploading GCS: 100%|██████████| 3.41G/3.41G [27:44<00:00, 2.05MB/s]

[22:54:52] Job State: JOB_STATE_SUCCEEDED
Job Completed Successfully!


## Step 8: Download and Parse Batch Prediction Results

In [8]:
def download_and_parse_batch_results(configs, job_status_data, test_df, output_parquet_path):
    project_id = configs.agent_settings.project_id
    bucket_name = configs.agent_settings.bucket_name
    
    gcs_output_dir = job_status_data.get("outputInfo", {}).get("gcsOutputDirectory")
    if not gcs_output_dir:
        raise ValueError("No gcsOutputDirectory found in job status data.")
        
    print(f"Output directory in GCS: {gcs_output_dir}")
    storage_client = storage.Client(project=project_id)
    bucket = storage_client.bucket(bucket_name.replace("gs://", ""))
    
    prefix = gcs_output_dir.replace(f"gs://{bucket.name}/", "").replace(f"gs://{bucket.name}", "")
    if prefix and not prefix.endswith("/"):
        prefix += "/"
        
    blobs = list(bucket.list_blobs(prefix=prefix))
    jsonl_blobs = [b for b in blobs if b.name.endswith(".jsonl")]
    
    if not jsonl_blobs:
        raise FileNotFoundError(f"No prediction JSONL files found in GCS prefix: {prefix}")
        
    print(f"Found {len(jsonl_blobs)} prediction JSONL files. Downloading and parsing...")
    
    parsed_data = []
    for blob in jsonl_blobs:
        local_temp = f"temp_3p_{os.path.basename(blob.name)}"
        blob.download_to_filename(local_temp)
        
        with open(local_temp, 'r') as f:
            for line in f:
                if not line.strip():
                    continue
                p = json.loads(line)
                
                resp_text = ""
                try:
                    if 'response' in p:
                        resp_text = p['response']['candidates'][0]['content']['parts'][0]['text']
                    elif 'predictions' in p:
                        resp_text = p['predictions'][0]['candidates'][0]['content']['parts'][0]['text']
                except Exception:
                    pass
                
                parsed = parse_response(resp_text)
                if isinstance(parsed, list):
                    parsed = parsed[0] if len(parsed) > 0 else {}
                elif not isinstance(parsed, dict):
                    parsed = {}
                
                pred_id = parsed.get("id")
                pred_k = parsed.get("k")
                pred_class = parsed.get("class") or parsed.get("class_label") or "Noise"
                pred_explanation = parsed.get("explanation") or ""
                
                req_id = None
                req_k = None
                try:
                    parts = p['request']['contents'][0]['parts']
                    for part in parts:
                        if 'text' in part and "Sample ID (sclk):" in part['text']:
                            text = part['text']
                            parts_split = text.split("|")
                            req_id = parts_split[0].replace("Sample ID (sclk):", "").strip()
                            if len(parts_split) > 1 and "k:" in parts_split[1]:
                                req_k = int(parts_split[1].replace("k:", "").strip())
                except Exception:
                    pass
                
                sclk_val = pred_id if pred_id else req_id
                k_val = pred_k if pred_k is not None else req_k
                
                if sclk_val is None:
                    continue
                    
                try:
                    sclk_val = str(int(float(sclk_val)))
                except Exception:
                    sclk_val = str(sclk_val)
                    
                try:
                    k_val = int(k_val)
                except Exception:
                    k_val = 0
                
                parsed_data.append({
                    "sclk": sclk_val,
                    "predicted_label": pred_class,
                    "explanation": pred_explanation,
                    "k": k_val
                })
        os.remove(local_temp)
        
    predictions_df = pd.DataFrame(parsed_data)
    test_lookup = test_df[['sclk', 'class']].copy()
    test_lookup['sclk'] = test_lookup['sclk'].astype(str)
    
    merged_df = pd.merge(predictions_df, test_lookup, on='sclk', how='inner')
    merged_df.rename(columns={"class": "true_label"}, inplace=True)
    merged_df = merged_df[['sclk', 'true_label', 'predicted_label', 'explanation', 'k']]
    
    os.makedirs(os.path.dirname(output_parquet_path), exist_ok=True)
    merged_df.to_parquet(output_parquet_path)
    print(f"Saved results to {output_parquet_path}")
    return merged_df

# 6. Download and Parse Stage 1 predictions
output_parquet_stage1 = os.path.join(output_dir, "results_3p_ablation_stage1.parquet")
stage1_df = download_and_parse_batch_results(configs, job_status, test_df, output_parquet_stage1)

# 7. Identify samples predicted as "3" or "3-P" for Stage 2 Self-Correction
reclass_candidates = stage1_df[stage1_df['predicted_label'].isin(["3", "3-P"])].copy()
print(f"\nFound {len(reclass_candidates)} candidates predicted as '3' or '3-P' for Stage 2 self-correction.")

if len(reclass_candidates) > 0:
    # Get corresponding rows with full spectrum data from test_df
    reclass_test_df = test_df[test_df['sclk'].astype(str).isin(reclass_candidates['sclk'].astype(str))].copy()

    # Merge Stage 1 predicted_label and explanation columns into reclass_test_df
    candidate_info = reclass_candidates[['sclk', 'predicted_label', 'explanation']].copy()
    candidate_info['sclk'] = candidate_info['sclk'].astype(str)
    reclass_test_df['sclk'] = reclass_test_df['sclk'].astype(str)
    reclass_test_df = pd.merge(reclass_test_df, candidate_info, on='sclk', how='inner')

    # Generate Stage 2 Requests JSONL
    local_jsonl_reclass = os.path.join(output_dir, "reclassify_requests_3p.jsonl")
    create_self_correction_batch_input_file(reclass_test_df, explanations_pool, local_jsonl_reclass)

    # Submit and poll Stage 2 job
    print("\n--- STAGE 2: VLM Self-Correction Prediction ---")
    job_status_reclass = submit_and_poll_batch_job(configs, local_jsonl_reclass)

    # Download and Parse Stage 2 results
    output_parquet_stage2 = os.path.join(output_dir, "results_3p_ablation_stage2.parquet")
    stage2_df = download_and_parse_batch_results(configs, job_status_reclass, reclass_test_df, output_parquet_stage2)

    # Merge refinement results back into predictions
    print("\nMerging refinement predictions...")
    refinement_map = dict(zip(stage2_df['sclk'].astype(str), stage2_df['predicted_label']))

    final_preds = []
    for _, row in stage1_df.iterrows():
        sclk_str = str(row['sclk'])
        pred = row['predicted_label']
        if sclk_str in refinement_map:
            refined_pred = refinement_map[sclk_str]
            print(f"  SCLK {sclk_str}: Refined prediction from {pred} -> {refined_pred}")
            pred = refined_pred
        final_preds.append(pred)

    stage1_df['predicted_label'] = final_preds

# Save Final Integrated Results
output_parquet = os.path.join(output_dir, "results_3p_ablation.parquet")
stage1_df.to_parquet(output_parquet)
print(f"Saved integrated hierarchical classification results to {output_parquet}")
results_df = stage1_df

Output directory in GCS: gs://turansgenaibb/output/prediction-model-2026-06-28T22:37:33.480927Z
Found 1 prediction JSONL files. Downloading and parsing...
Saved results to study_results_3p_self_correct/results_3p_ablation_stage1.parquet

Found 99 candidates predicted as '3' or '3-P' for Stage 2 self-correction.
Generating Stage 2 self-correction request file for 99 samples...
Pre-rendering and caching target spectrum images...
  -> Pre-rendered 99/99 images...
Generated JSONL file with 99 requests at study_results_3p_self_correct/reclassify_requests_3p.jsonl

--- STAGE 2: VLM Self-Correction Prediction ---
Uploading study_results_3p_self_correct/reclassify_requests_3p.jsonl to GCS bucket: turansgenaibb


Uploading GCS: 100%|██████████| 36.5M/36.5M [00:07<00:00, 4.34MB/s]

Uploaded to gs://turansgenaibb/input/reclassify_requests_3p.jsonl
Submitting Batch Job to Vertex AI...
Job Submitted! Name: projects/355771430623/locations/global/batchPredictionJobs/471239963426422784
Polling job status every 30 seconds...
[22:58:16] Job State: JOB_STATE_PENDING


Uploading GCS: 100%|██████████| 36.5M/36.5M [00:24<00:00, 4.34MB/s]

[22:58:48] Job State: JOB_STATE_QUEUED
[22:59:19] Job State: JOB_STATE_RUNNING
[22:59:50] Job State: JOB_STATE_RUNNING
[23:00:22] Job State: JOB_STATE_RUNNING
[23:00:53] Job State: JOB_STATE_RUNNING
[23:01:25] Job State: JOB_STATE_RUNNING


Uploading GCS: 100%|██████████| 36.5M/36.5M [03:52<00:00, 157kB/s] 

[23:01:56] Job State: JOB_STATE_SUCCEEDED
Job Completed Successfully!
Output directory in GCS: gs://turansgenaibb/output/prediction-model-2026-06-28T22:58:14.852523Z


Found 1 prediction JSONL files. Downloading and parsing...
Saved results to study_results_3p_self_correct/results_3p_ablation_stage2.parquet

Merging refinement predictions...
  SCLK 1519585197: Refined prediction from 3-P -> 3-P
  SCLK 1519584354: Refined prediction from 3-P -> 3
  SCLK 1543799120: Refined prediction from 3-P -> 3-P
  SCLK 1514157783: Refined prediction from 3-P -> 3-P
  SCLK 1521639847: Refined prediction from 3 -> 3
  SCLK 1509314528: Refined prediction from 3 -> 3
  SCLK 1557520217: Refined prediction from 3-P -> 3-P
  SCLK 1519560464: Refined prediction from 3 -> 3
  SCLK 1498504101: Refined prediction from 3 -> 3
  SCLK 1509338934: Refined prediction from 3 -> 3
  SCLK 1511733380: Refined prediction from 3-P -> 3-P
  SCLK 1521642075: Refined prediction from 3 -> 3
  SCLK 1511733538: Refined prediction from 3 -> 3
  SCLK 1489091121: Refined prediction from 3-P -> 3-P
  SCLK 1589074827: Refined prediction from 3 -> 3-P
  SCLK 1544895118: Refined prediction from 3-P

## Step 9: Evaluate Results and Plot Metrics

In [9]:
def generate_metrics_and_plots(results_df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    k_values = sorted(results_df['k'].unique())
    metrics_summary = {}
    
    print("\n" + "="*50 + "\n3-P focused Ablation Study Summary\n" + "="*50)
    
    for k in k_values:
        sub_df = results_df[results_df['k'] == k]
        
        # Overall accuracy
        overall_acc = accuracy_score(sub_df['true_label'], sub_df['predicted_label'])
        print(f"\nk = {k}-shot classification:")
        print(f"  Overall Accuracy: {overall_acc:.2%}")
        
        # Calculate confusion matrix for all classes
        labels = sorted(results_df['true_label'].unique())
        cm = confusion_matrix(sub_df['true_label'], sub_df['predicted_label'], labels=labels)
        
        # Plot Confusion Matrix
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
        plt.title(f"Confusion Matrix (k={k} shots) - Acc: {overall_acc:.2%}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        
        cm_path = os.path.join(output_dir, f"confusion_matrix_k{k}.png")
        plt.savefig(cm_path)
        plt.close()
        print(f"  Saved confusion matrix plot to {cm_path}")
        
        # Calculate class-specific metrics for 3-P
        target_cls = "3-P"
        
        # True Positives, False Positives, False Negatives, True Negatives
        y_true_binary = (sub_df['true_label'] == target_cls).astype(int)
        y_pred_binary = (sub_df['predicted_label'] == target_cls).astype(int)
        
        cm_binary = confusion_matrix(y_true_binary, y_pred_binary)
        if cm_binary.shape == (2, 2):
            tn, fp, fn, tp = cm_binary.ravel()
        else:
            tn, fp, fn, tp = 0, 0, 0, 0
            if len(y_true_binary.unique()) == 1:
                val = y_true_binary.iloc[0]
                if val == 0:
                    tn = len(y_true_binary)
                else:
                    tp = len(y_true_binary)
            
        precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        
        print(f"  Class 3-P Specific Metrics:")
        print(f"    TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")
        print(f"    Precision: {precision:.2%}")
        print(f"    Recall (Sensitivity): {recall:.2%}")
        print(f"    F1-score: {f1:.2%}")
        
        metrics_summary[str(k)] = {
            "overall_accuracy": float(overall_acc),
            "class_3p_metrics": {
                "tp": int(tp),
                "fp": int(fp),
                "fn": int(fn),
                "tn": int(tn),
                "precision": float(precision),
                "recall": float(recall),
                "f1_score": float(f1)
            }
        }
        
    metrics_json_path = os.path.join(output_dir, "metrics_3p_ablation_self_correct.json")
    with open(metrics_json_path, 'w') as f:
        json.dump(metrics_summary, f, indent=4)
    print(f"\nSaved metrics summary JSON to {metrics_json_path}")

generate_metrics_and_plots(results_df, output_dir)


3-P focused Ablation Study Summary

k = 4-shot classification:
  Overall Accuracy: 83.25%
  Saved confusion matrix plot to study_results_3p_self_correct/confusion_matrix_k4.png
  Class 3-P Specific Metrics:
    TP: 3, FP: 26, FN: 2, TN: 596
    Precision: 10.34%
    Recall (Sensitivity): 60.00%
    F1-score: 17.65%

Saved metrics summary JSON to study_results_3p_self_correct/metrics_3p_ablation_self_correct.json
